In [14]:
from torch.utils.data import DataLoader, Subset, TensorDataset, ConcatDataset, Dataset
from torchvision.transforms import AutoAugment, AutoAugmentPolicy
from torch.utils.tensorboard import SummaryWriter
from torchvision import datasets, transforms
from sklearn.metrics import accuracy_score
import torch.nn.functional as F
import matplotlib.pyplot as plt
import torch.optim as optim
from typing import List
from tqdm import tqdm
import torch.nn as nn
import numpy as np
import torchvision
import random
import torch
import os

In [7]:
class EarlyStopper:
    def __init__(self, patience=5, target_val=None):
        
        self.patience = patience
        self.counter =0
        self.min_validation_loss = float('inf')
        self.target =target_val
        
    def early_stop(self, validation_loss, accuracy):
             
        if self.target is not None and accuracy >= self.target:
            return True
        
        if validation_loss <= self.min_validation_loss:
            self.min_validation_loss = validation_loss
            self.counter = 0
            
        elif validation_loss <= self.min_validation_loss:
            self.counter += 1
            if self.counter >= self.patience:
                return True
        return False
            

In [8]:
def train(model, 
          dataloader, 
          optimizer,
          loss_function, 
          device, 
          clip_grad_norm=False):
    
    model.train()
    total_loss = 0.0
    
    for x, y in dataloader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        y_hat = model(x)
        loss = loss_function(y_hat, y)
        loss.backward()
        
        if clip_grad_norm:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        total_loss += loss.item()
        
    return total_loss / len(dataloader)

def test(model, 
         dataloader, 
         loss_function, 
         device ):
    correct = 0
    total_loss = 0
    total = 0
    
    model.eval()
    
    with torch.no_grad():
        for x, y in dataloader:
            x, y = x.to(device), y.to(device)
    
            y_hat = model(x)
            loss = loss_function(y_hat, y)
            total_loss += loss.item()
            
            pred = torch.argmax(y_hat, dim=1)
            correct += (pred == y).sum().item()
            total += y.size(0)
            
    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy

In [9]:

def train_model(
    logs_dir,
    model,
    train_dataloader,
    test_dataloader,
    epochs,
    model_kwargs=None,
    optimizer_func=None,
    loss_function= torch.nn.CrossEntropyLoss,
    device= torch.device("cpu"),
    use_early_stopper= False,
    clip_grad_norm=False,
    weight_init=False,
    use_scheduler=False,
    scheduler_kwargs={"mode":'min', "factor" :0.5, "patience": 5},
    early_stopping_patience=10,
    early_stopping_target=98.5,
):
    
    os.makedirs(f"{logs_dir}/logs", exist_ok=True)
    os.makedirs(f"{logs_dir}/results", exist_ok=True)
    writer = SummaryWriter(f"{logs_dir}/logs")

    if isinstance(model, type):  # class
        model_kwargs = model_kwargs or {}
        model = model(**model_kwargs).to(device)
    
        if weight_init is not None:
            model.weight_initializer(weight_init)
            
    else:  # instance
        model = model.to(device)

    optimizer = optimizer_func(model.parameters() )
    
    if use_early_stopper:
        early_stopper = EarlyStopper(
            patience=early_stopping_patience, 
            target_val=early_stopping_target )
    
    if use_scheduler:
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer,
             **scheduler_kwargs
          )

    history = {
        'train_loss': [],
        'test_loss': [],
        'test_accuracy': []
    }
 
    for epoch in tqdm(range(epochs), desc="Training"):
        train_loss = train(model, train_dataloader, optimizer, 
                           loss_function, device, clip_grad_norm)
        history['train_loss'].append(train_loss)
        writer.add_scalar(f'{logs_dir} Loss/train', train_loss, epoch)
        
        if test_dataloader is not None:
            test_loss, test_accuracy = test(model, test_dataloader, loss_function, device)
            history['test_loss'].append(test_loss)
            history['test_accuracy'].append(test_accuracy)
            
            writer.add_scalar(f'{logs_dir} Loss/test', test_loss, epoch)
            writer.add_scalar(f'{logs_dir} Accuracy/test', test_accuracy, epoch)

            if use_scheduler:
                scheduler.step(test_loss)
                
            if use_early_stopper:
                if early_stopper.early_stop(test_loss,accuracy=test_accuracy):
                    print("Early Stopper at epoch {} with accuracy {}% ".format(epoch, test_accuracy))
                    break
            
    writer.close()
    for key, values in history.items():
        if values:        
            np.save(f"{logs_dir}/results/{key}.npy", values)
    if test_dataloader is not None:
        return model, history['train_loss'], history['test_loss'], history['test_accuracy']
    else:
        return model, history['train_loss']

# Download Datasets
1. Create transforms through *AutoAugmentPolicy.CIFAR10*
2. Download the datasets  
3. Choose batch_size

In [12]:
batch_size = 256

transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.AutoAugment(policy=AutoAugmentPolicy.CIFAR10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.4914, 0.4822, 0.4465],
        std=[0.2470, 0.2435, 0.2616],
    ),
    
])  
trainset = torchvision.datasets.CIFAR10(
    root='./data', 
    train=True,
    download=True, 
    transform=transform
)
testset = torchvision.datasets.CIFAR10(
    root='./data', 
    train=True,
    download=True, 
    transform=transform
)
 
test_dataloader = DataLoader(testset, batch_size=batch_size, shuffle=True)
train_dataloader = DataLoader(trainset, batch_size=batch_size, shuffle=True)
 

100%|██████████████████████████████████████████████████████████████| 170498071/170498071 [06:06<00:00, 464572.78it/s]


Extracting ./data/cifar-10-python.tar.gz to ./data
Files already downloaded and verified


# Model

In [15]:
class Baseline(torch.nn.Module):
    def __init__(self, 
                 input_size: int, 
                 hidden_dims:List,
                 layers_per_block:List,
                 apply_pooling:List,
                 kernel_sizes:List,
                 apply_bn:List,
                 num_blocks:int=5,
                 num_classes:int=10, 
                 activation_function:torch.nn.Module=torch.nn.ReLU,
                 dropout_rate:float=0.3):
        super().__init__()

        assert len(apply_bn) == num_blocks
        assert len(apply_pooling) == num_blocks
        assert len(layers_per_block) == num_blocks
        assert len(hidden_dims)-2 == num_blocks

        self.activation_function = activation_function
        self.blocks = nn.ModuleList()
        in_channels = input_size
    
         
        for i in range(num_blocks):
            block = self.make_block(
                 in_channels=in_channels,
                 out_channels=hidden_dims[i],
                 num_layers=layers_per_block[i],
                 kernel_size=kernel_sizes[i],
                 dropout_rate=dropout_rate,
                 apply_bn=apply_bn[i],
                 apply_pooling=apply_pooling[i]
            )
            self.blocks.append(block)
            in_channels = hidden_dims[i]
        hidden_dims[i] = self.get_flat_size(input_size).shape[1]
        self.fc = nn.Sequential(
            nn.Linear(hidden_dims[i], hidden_dims[i+1]),
            activation_function(),
          
            nn.Linear(hidden_dims[i+1], hidden_dims[i+2]),
            activation_function(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_dims[i+2], num_classes)     
         )       
    def get_flat_size(self, input_size):
        dummy_input = torch.zeros(1, 3, input_size, input_size)
        for block in self.blocks:
            dummy_input = block(dummy_input)
        return torch.flatten(dummy_input, start_dim=1)
    def make_block(
        self,
        in_channels: int,
        out_channels: int,
        num_layers: int,
        kernel_size: int,
       
        apply_bn: bool,
        apply_pooling: bool,
         ) :
        
        layers = []
    
        for i in range(num_layers):
                 
            curr_in = in_channels if i == 0 else out_channels
            
            layers.append(nn.Conv2d(
                curr_in,
                out_channels,
                kernel_size=kernel_size,
                padding=kernel_size // 2,
                bias=False
            ))
            layers.append(self.activation_function())
            if apply_pooling:
                layers.append(nn.MaxPool2d(kernel_size=2, stride=2))
            if apply_bn:
                layers.append(nn.BatchNorm2d(out_channels))     
            
             
             
        return nn.Sequential(*layers)
        
    def forward(self, x):
        for block in self.blocks:
            x = block(x)
        flat_x = torch.flatten(x, start_dim=1)

        logits = self.fc(flat_x)
        return logits
    def count_parameters(self):
        total = 0
        for name, param in self.named_parameters():
            total += param.numel()
        return total
    def weight_initializer(self,initialization='he'):
        for module in self.modules():
            if isinstance(module, nn.Conv2d) or isinstance(module, nn.Linear):
                if initialization == 'normal':
                    nn.init.normal_(module.weight, mean=0.0, std=0.01) 
                elif initialization == 'xavier':
                    nn.init.xavier_normal_(module.weight)
                elif initialization == 'he':
                    nn.init.kaiming_normal_(module.weight)
                elif initialization == 'xavier_uniform':
                    nn.init.xavier_uniform_(module.weight)
                elif initialization == 'he_uniform':
                    nn.init.kaiming_uniform_(module.weight, mode='fan_in', nonlinearity='relu')
                    
            if hasattr(module, 'bias') and module.bias is not None:
                        nn.init.constant_(module.bias, 0)
        return self

   
    

        
         

# Losely base from 
Hertel, L., Barth, E., Käster, T., & Martinetz, T. (2015). Deep convolutional neural networks as generic feature extractors. In 2015 International Joint Conference on Neural Networks (IJCNN) (pp. 1–8). IEEE. https://doi.org/10.1109/IJCNN.2015.7280642

![Model Arquitecture base](model_architecture.png )